<a href="https://colab.research.google.com/github/poojabisht10/Deep-Learning/blob/main/Lab_Assignments/UCS761_Lab_6_Architecture_building.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
# for results to remain consistent
np.random.seed(42)

In [2]:
# generating input data (400 samples, 3 features)
X = np.random.uniform(-2, 2, (400,3))

# creating nonlinear target
y = (
    np.sin(X[:,0]) +
    0.5*(X[:,1]**2) -
    0.8*X[:,2]
)

y = y.reshape(-1,1)

# This function combines multiple patterns of behavior,
# which makes it difficult for a simple linear model to represent it accurately.

In [4]:
# ReLU
def relu(z):
    return np.maximum(0,z)

def drelu(z):
    return (z>0).astype(float)


# Sigmoid
def sig(z):
    return 1/(1+np.exp(-z))

def dsig(z):
    s = sig(z)
    return s*(1-s)


# Tanh
def th(z):
    return np.tanh(z)

def dth(z):
    return 1-np.tanh(z)**2


# Leaky ReLU
def lrelu(z,a=0.01):
    return np.where(z>0,z,a*z)

def dlrelu(z,a=0.01):
    return np.where(z>0,1,a)


# Softplus
def sp(z):
    return np.log(1+np.exp(z))

def dsp(z):
    return sig(z)

# we need derivatives so that gradients can flow backward during training

In [5]:
def init_p(dims):

    p = {}

    for i in range(1,len(dims)):

        p["W"+str(i)] = np.random.uniform(
            -0.5,0.5,(dims[i],dims[i-1])
        )

        p["b"+str(i)] = np.zeros((dims[i],1))

# initialize weights randomly so each neuron starts with different values
# set biases to zero initially; they will adjust during the training process

    return p

In [7]:
def fwd(X,p,act):

    A = X.T
    c = []
    L = len(p)//2

    for i in range(1,L):

        W = p["W"+str(i)]
        b = p["b"+str(i)]

        Z = W@A + b
        A = act(Z)

        c.append((A,Z,W))

    # output layer (linear)
    W = p["W"+str(L)]
    b = p["b"+str(L)]

    Z = W@A + b
    A = Z

    c.append((A,Z,W))

    return A,c

# forward pass simply moves data through layers step by step

In [8]:
def loss(yh,y):

    l = np.mean((yh.T-y)**2)

# mean squared error is used because this problem is treated as regression
# larger prediction errors contribute more strongly to the loss

    return l

In [9]:
def bwd(X,y,p,c,dact):

    g = {}
    m = X.shape[0]
    L = len(p)//2

    yh = c[-1][0]

    dA = 2*(yh.T-y).T/m

    for i in reversed(range(1,L+1)):

        A_prev = X.T if i==1 else c[i-2][0]
        A,Z,W = c[i-1]

        if i!=L:
            dZ = dA*dact(Z)
        else:
            dZ = dA

        g["dW"+str(i)] = dZ@A_prev.T
        g["db"+str(i)] = np.sum(dZ,axis=1,keepdims=True)

        dA = W.T@dZ

    return g

In [11]:
def upd(p,g,lr):

    L = len(p)//2

    for i in range(1,L+1):

        p["W"+str(i)] -= lr*g["dW"+str(i)]
        p["b"+str(i)] -= lr*g["db"+str(i)]

# parameters are updated in the opposite direction of the gradient
# this step-by-step adjustment helps minimize the loss

    return p

In [12]:
def train(X,y,dims,act,dact,lr=0.01,ep=1000):

    p = init_p(dims)

    for e in range(ep):

        yh,c = fwd(X,p,act)

        l = loss(yh,y)

        g = bwd(X,y,p,c,dact)

        p = upd(p,g,lr)

        if e%200==0:
            print("ep:",e,"loss:",l)

    return p,g

In [13]:
# model structures

A = [3,4,1]

B = [3,6,6,1]

C = [3,8,8,8,8,1]

D = [3,8,8,8,8,8,8,8,8,1]

# deeper networks perform multiple layers of transformations
# this can help capture more complex patterns in the data
# however, it can also make

In [ ]:
p,g = train(X,y,A,relu,drelu)

ep: 0 loss: 3.219058085818201
ep: 200 loss: 0.49268594052687364
ep: 400 loss: 0.31953875797866105
ep: 600 loss: 0.19648752020178062
ep: 800 loss: 0.13524070487719164


In [15]:
p,g = train(X,y,A,relu,drelu)
g1 = np.linalg.norm(g["dW1"])
gL = np.linalg.norm(g["dW"+str(len(p)//2)])

print("grad norm L1:",g1)
print("grad norm last:",gL)

# comparing these tells us how gradients are behaving across different layers

ep: 0 loss: 3.219058085818201
ep: 200 loss: 0.49268594052687364
ep: 400 loss: 0.31953875797866105
ep: 600 loss: 0.19648752020178062
ep: 800 loss: 0.13524070487719164
grad norm L1: 0.04521736074874136
grad norm last: 0.03727266746906128
